# Klasifikasi Lesi Kulit ISIC2018 Task 3 — CNN dengan Residual Connection (ResNet)

Alur pipeline:
1. **Configuration** — semua path dataset & hyperparameter dikumpulkan di satu section awal untuk mudah diubah.
2. Memuat ground truth resmi ISIC 2018 Task 3 & mengonversi *one-hot* ke label tunggal.
3. **EDA** — distribusi kelas, ukuran/channel gambar, statistik piksel per channel.
4. *Stratified split* train/validation dari training set resmi (test & validation resmi disisihkan penuh untuk evaluasi akhir).
5. Balancing opsional via downsampling kelas mayoritas.
6. Augmentasi + normalisasi ImageNet, dataset & DataLoader.
7. Model **ResNet-18 pretrained ImageNet** dengan head 7 kelas.
8. Training baseline, eksperimen hyperparameter (LR, batch size, dropout), ringkasan tabel, dan evaluasi akhir di test set resmi.

> Struktur data mengikuti rilis resmi ISIC 2018 Challenge (bukan versi Kaggle). Validation_Input resmi tidak memiliki ground truth publik sehingga tidak dipakai untuk evaluasi.

## 1. Configuration — path dataset & semua hyperparameter

> **Ubah nilai di sini.** Semua path **terdeteksi otomatis**: di Kaggle diambil dari `/kaggle/input` (folder yang memuat `ISIC2018_Task3_Training_Input` dipilih otomatis), di lokal memakai folder `dataset/`. Semua output (model, history, laporan, gambar) **disimpan** ke `CONFIG['output_dir']`.

In [ ]:
# =========================================================================
# SECTION 1: CONFIGURATION (SEMUA hyperparameter & path DIKUMPULKAN DI SINI)
# =========================================================================
import os
import glob
from pathlib import Path

CONFIG = {
    # ================= PATH DATASET =================
    # Dataset mengikuti rilis resmi ISIC 2018 Task 3.
    # - KAGGLE : path otomatis dari /kaggle/input (folder berisi
    #            ISIC2018_Task3_Training_Input dipilih otomatis).
    # - LOKAL  : pakai folder relatif "dataset".
    # Boleh juga eksplisit, misal "/kaggle/input/isic2018-task3".
    "data_dir": "dataset",

    # Nama subfolder/CSV relatif di dalam data_dir (mengikuti rilis resmi).
    "train_img_dir": "ISIC2018_Task3_Training_Input",
    "test_img_dir":  "ISIC2018_Task3_Test_Input",
    "val_img_dir":   "ISIC2018_Task3_Validation_Input",
    "train_gt_dir":  "ISIC2018_Task3_Training_GroundTruth",
    "test_gt_dir":   "ISIC2018_Task3_Test_GroundTruth",
    "val_gt_dir":    "ISIC2018_Task3_Validation_GroundTruth",

    # ================= OUTPUT (disimpan ke folder ini) =================
    # Di Kaggle cwd = /kaggle/working. Semua hasil (format file) tersimpan.
    "output_dir": "output",
    "save_model": True,     # simpan checkpoint state_dict model terbaik per run
    "save_history": True,   # simpan history loss/acc per run (JSON)

    # ================= DATA =================
    "class_columns": ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"],
    "num_classes": 7,
    "img_size": 224,            # resize input (ResNet menerima 224x224)
    "val_ratio": 0.15,          # stratified split dari training set
    "random_state": 42,
    "balance_threshold": 500,   # batas downsampling kelas mayoritas (opsional)
    "sample_size": 300,         # sampel EDA (naikkan kalau waktu memungkinkan)

    # ================= IMAGE NORMALIZATION (pretrained ImageNet) =================
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std":  [0.229, 0.224, 0.225],

    # ================= TRAINING (baseline) =================
    "batch_size": 32,
    "lr": 1e-4,
    "dropout": 0.3,
    "optimizer_name": "adam",   # 'adam' | 'sgd' | 'rmsprop'
    "weight_decay": 1e-4,
    "num_epochs": 20,
    "experiment_epochs": 15,    # epoch untuk tiap eksperimen hyperparameter

    # ================= EKSPERIMEN HYPERPARAMETER (nilai tunggal, ubah manual) =================
    # Ubah nilai di bawah ini lalu jalankan ulang cell eksperimen yang bersangkutan (Section 15).
    "lr_experiment": 1e-3,          # coba nilai LR berbeda dari baseline
    "batch_size_experiment": 16,    # coba batch size berbeda dari baseline
    "dropout_experiment": 0.2,      # coba dropout berbeda dari baseline

    # ================= LINGKUNGAN =================
    "device": "auto",           # 'auto' | 'cuda' | 'cpu'
    "num_workers": 0,           # 0 aman untuk Windows; di Kaggle boleh 2-4
    "pin_memory": True,
}

# ---- Resolusi path: otomatis cari dataset di /kaggle/input, selain itu pakai data_dir ----
OUTPUT_DIR    = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _resolve_data_root(configured):
    # Utamakan path eksplisit. Di Kaggle, cari otomatis folder /kaggle/input
    # yang memuat ISIC2018_Task3_Training_Input.
    explicit = str(configured)
    if os.path.isdir(os.path.join(explicit, CONFIG["train_img_dir"])):
        return explicit
    if os.path.isdir("/kaggle/input"):
        for entry in sorted(os.listdir("/kaggle/input")):
            cand = os.path.join("/kaggle/input", entry)
            if os.path.isdir(os.path.join(cand, CONFIG["train_img_dir"])):
                return cand
    return explicit

DATA_DIR = _resolve_data_root(CONFIG["data_dir"])

TRAIN_IMG_DIR = os.path.join(DATA_DIR, CONFIG["train_img_dir"])
TEST_IMG_DIR  = os.path.join(DATA_DIR, CONFIG["test_img_dir"])
VAL_IMG_DIR   = os.path.join(DATA_DIR, CONFIG["val_img_dir"])

# Ground truth berbentuk CSV di dalam folder *_GroundTruth.
def _first_csv(folder):
    hits = glob.glob(os.path.join(folder, "*.csv"))
    return hits[0] if hits else None

TRAIN_GT_PATH = _first_csv(os.path.join(DATA_DIR, CONFIG["train_gt_dir"]))
TEST_GT_PATH  = _first_csv(os.path.join(DATA_DIR, CONFIG["test_gt_dir"]))
VAL_GT_PATH   = _first_csv(os.path.join(DATA_DIR, CONFIG["val_gt_dir"]))

for p, desc in [(TRAIN_IMG_DIR, "Training input"), (TEST_IMG_DIR, "Test input"),
                (VAL_IMG_DIR, "Validation input")]:
    assert os.path.isdir(p), f"{desc}: folder tidak ditemukan -> {p}"
for p, desc in [(TRAIN_GT_PATH, "Training ground truth"), (TEST_GT_PATH, "Test ground truth"),
                (VAL_GT_PATH, "Validation ground truth")]:
    assert p and os.path.isfile(p), f"{desc}: file tidak ditemukan -> {p}"

IMG_SIZE  = CONFIG["img_size"]
CLASS_COLUMNS = CONFIG["class_columns"]

print("Data directory   :", DATA_DIR)
print("Output directory :", OUTPUT_DIR)
print("Train gambar     :", len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))))
print("Test gambar      :", len(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))))
print("Validation gambar:", len(glob.glob(os.path.join(VAL_IMG_DIR, "*.jpg"))))
print("Semua path dataset valid. Config dimuat; DEVICE di-set di Section 2 (setelah import torch).")

## 2. Imports

In [ ]:
# =========================================================================
# SECTION: IMPORTS
# =========================================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# =========================================================================
# DEVICE (dipindah ke sini karena butuh torch)
# =========================================================================
if CONFIG["device"] == "cpu":
    device = torch.device("cpu")
elif CONFIG["device"] == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA diminta tapi tidak tersedia.")
    device = torch.device("cuda")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang dipakai:", device)

print("PyTorch", torch.__version__, "| device:", device)
print("Data train:", len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))),
      "| Data test:", len(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))))

## 3. Muat Ground Truth (one-hot → label tunggal)

In [ ]:
# =========================================================================
# SECTION 2: Konversi Ground Truth One-Hot ke Label Tunggal
# Ground truth resmi ISIC2018 berformat one-hot (kolom MEL, NV, BCC,
# AKIEC, BKL, DF, VASC, isinya 1.0 di kolom kelas yang benar).
# =========================================================================

def onehot_to_label(df_raw):
    df = df_raw.copy()
    df["dx"] = df[CLASS_COLUMNS].idxmax(axis=1)
    return df[["image", "dx"]]


train_gt_raw = pd.read_csv(TRAIN_GT_PATH)
test_gt_raw  = pd.read_csv(TEST_GT_PATH)
val_gt_raw   = pd.read_csv(VAL_GT_PATH)
print("Contoh ground truth training:")
print(train_gt_raw.head())

train_df_full = onehot_to_label(train_gt_raw)
test_df  = onehot_to_label(test_gt_raw)
val_df_official = onehot_to_label(val_gt_raw)

train_df_full["path"] = train_df_full["image"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x + ".jpg"))
test_df["path"]       = test_df["image"].apply(lambda x: os.path.join(TEST_IMG_DIR, x + ".jpg"))
val_df_official["path"] = val_df_official["image"].apply(lambda x: os.path.join(VAL_IMG_DIR, x + ".jpg"))

print("\nJumlah data training (sebelum split val):", len(train_df_full))
print("Jumlah data test:", len(test_df))
print("Jumlah data validation resmi (berlabel):", len(val_df_official))

## 4. EDA — Distribusi Kelas

In [ ]:
# =========================================================================
# SECTION 3: EDA — Class Distribution
# =========================================================================

def class_distribution_table(df, name):
    counts = df["dx"].value_counts()
    pct = df["dx"].value_counts(normalize=True) * 100
    table = pd.DataFrame({"Jumlah": counts, "Persentase (%)": pct.round(2)})
    table.index.name = f"Kelas ({name})"
    return table


train_dist = class_distribution_table(train_df_full, "Training")
test_dist  = class_distribution_table(test_df, "Test")
val_dist   = class_distribution_table(val_df_official, "Validation Resmi")

print("Distribusi kelas — Training set:")
print(train_dist)
print("\nDistribusi kelas — Test set:")
print(test_dist)
print("\nDistribusi kelas — Validation set resmi:")
print(val_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_df_full["dx"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi Kelas — Training Set")
axes[0].set_xlabel("Kelas")
axes[0].set_ylabel("Jumlah Gambar")

test_df["dx"].value_counts().plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Distribusi Kelas — Test Set")
axes[1].set_xlabel("Kelas")
axes[1].set_ylabel("Jumlah Gambar")
plt.tight_layout()
plt.show()

## 5. EDA — Karakteristik Data (ukuran, channel, statistik piksel)

In [ ]:
# =========================================================================
# SECTION 4: EDA — Data Characteristics
# Dihitung dari sampel acak biar cepat (naikkan CONFIG["sample_size"] kalau
# waktu memungkinkan). Mengonfirmasi bahwa gambar tidak seragam ukurannya
# sehingga resize wajib dilakukan sebelum masuk model.
# =========================================================================

SAMPLE_SIZE = CONFIG["sample_size"]
sample_paths = train_df_full["path"].sample(n=SAMPLE_SIZE, random_state=CONFIG["random_state"]).tolist()

image_stats = []
for p in sample_paths:
    with Image.open(p) as img:
        width, height = img.size
        mode = img.mode          # 'RGB', 'L', 'RGBA', dst
        n_channels = len(img.getbands())
        file_size_kb = os.path.getsize(p) / 1024
    image_stats.append({
        "width": width, "height": height,
        "aspect_ratio": round(width / height, 3),
        "mode": mode, "n_channels": n_channels,
        "file_size_kb": round(file_size_kb, 1)
    })

stats_df = pd.DataFrame(image_stats)
print(stats_df.head())

print("\n=== Ukuran Gambar ===")
print(f"Resolusi unik: {stats_df[['width','height']].drop_duplicates().shape[0]} variasi dari {SAMPLE_SIZE} sampel")
print(f"Width  -> min: {stats_df['width'].min()}, max: {stats_df['width'].max()}, modus: {stats_df['width'].mode()[0]}")
print(f"Height -> min: {stats_df['height'].min()}, max: {stats_df['height'].max()}, modus: {stats_df['height'].mode()[0]}")

print("\n=== Channel Warna ===")
print(stats_df["mode"].value_counts())
print(f"Semua gambar RGB (3 channel)? {(stats_df['n_channels'] == 3).all()}")

print("\n=== Descriptive Statistics (width, height, aspect ratio, file size) ===")
print(stats_df[["width", "height", "aspect_ratio", "file_size_kb"]].describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(stats_df["width"], bins=20, color="steelblue", edgecolor="black")
axes[0].set_title("Distribusi Lebar Gambar (px)")
axes[0].set_xlabel("Width")

axes[1].hist(stats_df["height"], bins=20, color="seagreen", edgecolor="black")
axes[1].set_title("Distribusi Tinggi Gambar (px)")
axes[1].set_xlabel("Height")

axes[2].hist(stats_df["file_size_kb"], bins=20, color="indianred", edgecolor="black")
axes[2].set_title("Distribusi Ukuran File (KB)")
axes[2].set_xlabel("File Size (KB)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Statistik piksel per channel (subset kecil, baca penuh piksel lebih berat) ---
pixel_means, pixel_stds = [], []
for p in sample_paths[:100]:
    with Image.open(p) as img:
        arr = np.array(img.convert("RGB")) / 255.0
        pixel_means.append(arr.reshape(-1, 3).mean(axis=0))
        pixel_stds.append(arr.reshape(-1, 3).std(axis=0))

pixel_means = np.array(pixel_means)
pixel_stds  = np.array(pixel_stds)

pixel_stat_df = pd.DataFrame({
    "Channel": ["R", "G", "B"],
    "Mean": pixel_means.mean(axis=0).round(4),
    "Std":  pixel_stds.mean(axis=0).round(4)
})
print("\nStatistik pixel dataset (skala 0-1):")
print(pixel_stat_df)
print("\nStatistik ImageNet (dipakai untuk normalisasi karena pretrained):")
print(f"Mean: {CONFIG['imagenet_mean']}, Std: {CONFIG['imagenet_std']}")

## 6. Split Train / Validation & Mapping Label

In [ ]:
# =========================================================================
# SECTION 5: Split Train / Validation (dari Training set resmi)
# Test set resmi ISIC2018 (dengan ground truth) disisihkan penuh untuk
# evaluasi akhir, tidak disentuh selama development.
# =========================================================================

train_classes = sorted(train_df_full["dx"].unique())
class_to_idx = {c: i for i, c in enumerate(train_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

train_df_full["label"] = train_df_full["dx"].map(class_to_idx)
test_df["label"]       = test_df["dx"].map(class_to_idx)

train_df, val_df = train_test_split(
    train_df_full, test_size=CONFIG["val_ratio"],
    stratify=train_df_full["label"], random_state=CONFIG["random_state"]
)

print("Mapping kelas:", class_to_idx)
print("Jumlah data train:", len(train_df))
print("Jumlah data validation:", len(val_df))
print("Jumlah data test (resmi):", len(test_df))

## 7. Balancing Data (opsional, mengikuti strategi paper)

In [ ]:
# =========================================================================
# SECTION 6: Data Balancing (opsional)
# Threshold dari CONFIG["balance_threshold"] (default 500): downsampling
# kelas mayoritas (NV, MEL, BKL). Kelas minoritas (AKIEC, BCC, VASC, DF)
# diperkuat lewat augmentasi saat training, bukan oversampling di sini.
# =========================================================================

BALANCE_THRESHOLD = CONFIG["balance_threshold"]


def downsample_majority(df, threshold, seed):
    balanced_parts = []
    for cls, group in df.groupby("dx"):
        if len(group) > threshold:
            group = group.sample(n=threshold, random_state=seed)
        balanced_parts.append(group)
    return pd.concat(balanced_parts).reset_index(drop=True)


train_df_balanced = downsample_majority(train_df, BALANCE_THRESHOLD, CONFIG["random_state"])

print("Distribusi kelas training SEBELUM downsampling:")
print(train_df["dx"].value_counts())
print("\nDistribusi kelas training SETELAH downsampling:")
print(train_df_balanced["dx"].value_counts())

## 8. Transform & Augmentasi

In [ ]:
# =========================================================================
# SECTION 7: Transform dan Augmentasi
# Resize ke CONFIG["img_size"] (224x224, input ResNet-18), normalisasi pakai
# statistik ImageNet karena pakai pretrained weights. Augmentasi hanya
# diterapkan ke data training.
# =========================================================================

IMAGENET_MEAN = CONFIG["imagenet_mean"]
IMAGENET_STD  = CONFIG["imagenet_std"]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 9. Custom Dataset

In [ ]:
# =========================================================================
# SECTION 8: Custom Dataset
# =========================================================================

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

## 10. Model ResNet-18

In [ ]:
# =========================================================================
# SECTION 9: Model ResNet-18
# Memakai bobot pretrained ImageNet lalu mengganti fully connected layer
# terakhir jadi 7 kelas. dropout sebagai hyperparameter untuk eksperimen.
# =========================================================================

def build_model(num_classes=CONFIG["num_classes"], dropout=CONFIG["dropout"], pretrained=True):
    model = torchvision.models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    return model.to(device)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# sanity check arsitektur
m = build_model()
tot, tr = count_params(m)
print(f"ResNet-18 param: {tot/1e6:.2f}M total, {tr/1e6:.2f}M trainable")

## 11. Fungsi Training & Evaluasi per Epoch

In [ ]:
# =========================================================================
# SECTION 10: Fungsi Training dan Evaluasi per Epoch
# =========================================================================

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels

## 12. Fungsi Utama Training (hyperparameter sebagai argumen)

In [ ]:
# =========================================================================
# SECTION 11: Fungsi Utama Training
# Satu fungsi menerima kombinasi hyperparameter sebagai argumen, jadi tinggal
# dipanggil ulang untuk tiap eksperimen tanpa duplikasi kode. Default pakai
# train_df_balanced (hasil downsampling); ganti ke train_df kalau tanpa balancing.
# =========================================================================

def run_training(lr=CONFIG["lr"], batch_size=CONFIG["batch_size"], dropout=CONFIG["dropout"],
                 optimizer_name=CONFIG["optimizer_name"], weight_decay=CONFIG["weight_decay"],
                 num_epochs=CONFIG["num_epochs"], run_name="baseline",
                 train_data=None):

    if train_data is None:
        train_data = train_df_balanced

    train_loader = DataLoader(
        ISICDataset(train_data, train_transform),
        batch_size=batch_size, shuffle=True,
        num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
    )
    val_loader = DataLoader(
        ISICDataset(val_df, eval_transform),
        batch_size=batch_size, shuffle=False,
        num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
    )

    model = build_model(dropout=dropout)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Optimizer '{optimizer_name}' tidak dikenali")

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"[{run_name}] Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    history["best_val_acc"] = best_val_acc

    # Simpan state_dict model terbaik (val acc tertinggi sepanjang training)
    if CONFIG["save_model"] and best_state is not None:
        model_path = OUTPUT_DIR / f"model_{run_name}_best.pt"
        torch.save(best_state, str(model_path))
        print(f"  saved model {run_name} (val_acc {best_val_acc:.4f}) -> {model_path}")
    # Simpan history beserta config yang dipakai agar run bisa direproduksi
    if CONFIG["save_history"]:
        hist_path = OUTPUT_DIR / f"history_{run_name}.json"
        with open(hist_path, "w", encoding="utf-8") as f:
            json.dump({
                "run_name": run_name,
                "config": {k: v for k, v in CONFIG.items() if not k.startswith("_")},
                "history": history,
            }, f, indent=2)
        print(f"  saved history {run_name} -> {hist_path}")

    return model, history

## 13. Plot Kurva Loss & Accuracy

In [ ]:
# =========================================================================
# SECTION 13: Plot Kurva Loss dan Accuracy
# =========================================================================

def plot_history(history, title="Training History", fname=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title(f"{title} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train Acc")
    axes[1].plot(history["val_acc"], label="Val Acc")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    if fname:
        fig.savefig(str(OUTPUT_DIR / fname), bbox_inches="tight", dpi=150)
        print(f"  saved figure -> {OUTPUT_DIR / fname}")
    plt.show()

## 14. Jalankan Training Baseline

In [ ]:
# =========================================================================
# SECTION 12: Jalankan Training Baseline
# Baseline dulu pakai setting default (CONFIG), jadi patokan sebelum tuning.
# =========================================================================

baseline_model, baseline_history = run_training(
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"], dropout=CONFIG["dropout"],
    optimizer_name=CONFIG["optimizer_name"], weight_decay=CONFIG["weight_decay"],
    num_epochs=CONFIG["num_epochs"], run_name="baseline"
)

plot_history(baseline_history, "Baseline", fname="history_baseline.png")

## 15. Eksperimen Hyperparameter

Di sini diamati **satu per satu** (tanpa array/pengulangan): ganti nilai `CONFIG['lr_experiment']`, `CONFIG['batch_size_experiment']`, `CONFIG['dropout_experiment']` di Section 1 lalu jalankan ulang cell yang bersangkutan. Hasil tiap eksperimen disimpan ke `exp_results` dan dirangkum di Section 16.

In [ ]:
# Penampung hasil eksperimen untuk tabel ringkasan
exp_results = {}

In [ ]:
# --- 15a. Eksperimen Learning Rate ---
lr_val = CONFIG["lr_experiment"]
print(f"\n=== Eksperimen Learning Rate = {lr_val} ===")
_, hist_lr = run_training(
    lr=lr_val, batch_size=CONFIG["batch_size"], dropout=CONFIG["dropout"],
    optimizer_name="adam", num_epochs=CONFIG["experiment_epochs"], run_name=f"lr_{lr_val}"
)
print(f"LR={lr_val} -> Val Acc akhir: {hist_lr['val_acc'][-1]:.4f}, "
      f"Val Loss akhir: {hist_lr['val_loss'][-1]:.4f}")

exp_results["Learning Rate"] = {
    "Nilai": lr_val,
    "Val Acc": hist_lr["val_acc"][-1],
    "Val Loss": hist_lr["val_loss"][-1],
}

In [ ]:
# --- 15b. Eksperimen Batch Size ---
bs_val = CONFIG["batch_size_experiment"]
print(f"\n=== Eksperimen Batch Size = {bs_val} ===")
_, hist_bs = run_training(
    lr=CONFIG["lr"], batch_size=bs_val, dropout=CONFIG["dropout"],
    optimizer_name="adam", num_epochs=CONFIG["experiment_epochs"], run_name=f"bs_{bs_val}"
)
print(f"Batch Size={bs_val} -> Val Acc akhir: {hist_bs['val_acc'][-1]:.4f}, "
      f"Val Loss akhir: {hist_bs['val_loss'][-1]:.4f}")

exp_results["Batch Size"] = {
    "Nilai": bs_val,
    "Val Acc": hist_bs["val_acc"][-1],
    "Val Loss": hist_bs["val_loss"][-1],
}

In [ ]:
# --- 15c. Eksperimen Dropout Rate ---
dp_val = CONFIG["dropout_experiment"]
print(f"\n=== Eksperimen Dropout = {dp_val} ===")
_, hist_dp = run_training(
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"], dropout=dp_val,
    optimizer_name="adam", num_epochs=CONFIG["experiment_epochs"], run_name=f"dropout_{dp_val}"
)
print(f"Dropout={dp_val} -> Val Acc akhir: {hist_dp['val_acc'][-1]:.4f}, "
      f"Val Loss akhir: {hist_dp['val_loss'][-1]:.4f}")

exp_results["Dropout"] = {
    "Nilai": dp_val,
    "Val Acc": hist_dp["val_acc"][-1],
    "Val Loss": hist_dp["val_loss"][-1],
}

## 16. Ringkasan Hasil Eksperimen (tabel laporan)

In [ ]:
# =========================================================================
# SECTION 15: Rangkum Hasil Eksperimen dalam Tabel
# Nilai yang dirangkum adalah hasil re-run terakhir tiap eksperimen.
# =========================================================================

summary_df = pd.DataFrame(
    [{"Eksperimen": k, **v} for k, v in exp_results.items()]
)
print(summary_df)

# Simpan ringkasan eksperimen ke CSV
summary_path = OUTPUT_DIR / "hyperparameter_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\nsaved summary -> {summary_path}")

## 17. Evaluasi Akhir di Test Set Resmi

Pakai `baseline_model` atau model hasil kombinasi hyperparameter terbaik. Test set resmi ISIC2018 baru disentuh di sini — tidak pernah dilihat model selama training/tuning.

In [ ]:
# =========================================================================
# SECTION 16: Evaluasi Akhir di Test Set Resmi
# =========================================================================

# Gunakan model terbaik (mis. hasil eksperimen) sebagai best_model
best_model = baseline_model
test_loader = DataLoader(
    ISICDataset(test_df, eval_transform), batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
)
criterion = nn.CrossEntropyLoss()

test_loss, test_acc, preds, labels = evaluate(best_model, test_loader, criterion)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=train_classes))

# ---- Simpan output evaluasi ----
metrics_df = pd.DataFrame({"test_loss": [test_loss], "test_acc": [test_acc]})
metrics_path = OUTPUT_DIR / "test_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print(f"\nsaved metrics -> {metrics_path}")

report_dict = classification_report(labels, preds, target_names=train_classes,
                                    output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
report_path = OUTPUT_DIR / "test_classification_report.csv"
report_df.to_csv(report_path)
print(f"saved classification report -> {report_path}")

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=train_classes, yticklabels=train_classes,
            cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - Test Set Resmi ISIC2018")
plt.tight_layout()
cm_path = OUTPUT_DIR / "confusion_matrix_test.png"
fig.savefig(str(cm_path), bbox_inches="tight", dpi=150)
print(f"saved confusion matrix -> {cm_path}")
plt.show()

# Simpan daftar prediksi per-gambar untuk laporan
pred_df = test_df[["image", "dx"]].copy()
pred_df.rename(columns={"dx": "true_label"}, inplace=True)
pred_df["pred_class"] = [train_classes[p] for p in preds]
pred_df["correct"] = (pred_df["true_label"] == pred_df["pred_class"]).astype(int)
pred_path = OUTPUT_DIR / "test_predictions.csv"
pred_df.to_csv(pred_path, index=False)
print(f"saved predictions -> {pred_path}")

## 18. Evaluasi di Validation Set Resmi

Validation ground truth resmi sekarang tersedia (193 gambar berlabel). Evaluasi model terbaik yang sama seperti pada test set, untuk melihat konsistensi performa di data validasi resmi (yang memang dipisahkan panitia, bukan bagian dari training).

In [ ]:
# =========================================================================
# SECTION 17: Evaluasi di Validation Set Resmi (Ground Truth Resmi)
# =========================================================================

val_loader = DataLoader(
    ISICDataset(val_df_official, eval_transform), batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
)

val_loss, val_acc, val_preds, val_labels = evaluate(best_model, val_loader, criterion)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

print("\nClassification Report - Validation Resmi:")
print(classification_report(val_labels, val_preds, target_names=train_classes))

# Simpan metrik & laporan klasifikasi validation set
val_metrics_path = OUTPUT_DIR / "validation_metrics.csv"
pd.DataFrame({"val_loss": [val_loss], "val_acc": [val_acc]}).to_csv(val_metrics_path, index=False)
print(f"\nsaved metrics -> {val_metrics_path}")

val_report_path = OUTPUT_DIR / "validation_classification_report.csv"
val_report = pd.DataFrame(classification_report(val_labels, val_preds, target_names=train_classes,
                                                output_dict=True, zero_division=0)).T
val_report.to_csv(val_report_path)
print(f"saved classification report -> {val_report_path}")

cm_val = confusion_matrix(val_labels, val_preds)
fig_val, ax_val = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_val, annot=True, fmt="d", xticklabels=train_classes, yticklabels=train_classes,
            cmap="Purples", ax=ax_val)
ax_val.set_xlabel("Predicted")
ax_val.set_ylabel("Actual")
ax_val.set_title("Confusion Matrix - Validation Set Resmi ISIC2018")
plt.tight_layout()
cm_val_path = OUTPUT_DIR / "confusion_matrix_validation.png"
fig_val.savefig(str(cm_val_path), bbox_inches="tight", dpi=150)
print(f"saved confusion matrix -> {cm_val_path}")
plt.show()

# Simpan prediksi per-gambar pada validation set
val_pred_df = val_df_official[["image", "dx"]].copy()
val_pred_df.rename(columns={"dx": "true_label"}, inplace=True)
val_pred_df["pred_class"] = [train_classes[p] for p in val_preds]
val_pred_df["correct"] = (val_pred_df["true_label"] == val_pred_df["pred_class"]).astype(int)
val_pred_path = OUTPUT_DIR / "validation_predictions.csv"
val_pred_df.to_csv(val_pred_path, index=False)
print(f"saved predictions -> {val_pred_path}")

## Catatan

- **Upload ke Kaggle**: attach dataset berisi folder standar ISIC 2018 Task 3 (input + ground truth). Path dataset **terdeteksi otomatis** dari `/kaggle/input`; jika struktur sama persis, tidak perlu ubah apa pun. Bila nama root dataset berbeda, langsung tambahkan `'/kaggle/input/<nama-dataset>'` ke `CONFIG['data_dir']`.
- **Semua output tersimpan** ke `CONFIG['output_dir']` (`output/`): checkpoint model terbaik (`model_*_best.pt`), history JSON (`history_*.json`), kurva training (PNG), ringkasan eksperimen CSV, dan hasil evaluasi di test set serta validation set resmi (metrics CSV, classification report CSV, confusion matrix PNG, prediksi per-gambar CSV). Di Kaggle folder ini ada di `/kaggle/working/output/`.
- **Semua hyperparameter** (path, ukuran gambar, split, threshold balancing, batch size, learning rate, dropout, optimizer, weight decay, jumlah epoch, **nilai eksperimen**) dikumpulkan di **Section 1 (CONFIG)** — ubah di satu tempat saja.
- **Eksperimen memakai nilai tunggal**: ganti `CONFIG['lr_experiment']`, `CONFIG['batch_size_experiment']`, atau `CONFIG['dropout_experiment']` di Section 1 lalu jalankan ulang cell eksperimen yang bersangkutan (Section 15). Tidak ada array/pengulangan; hasil re-run terbaru otomatis menggantikan nilai sebelumnya di tabel ringkasan (Section 16).
- **Validation selama training** tetap memakai stratified split dari training set (agar pembanding eksperimen konsisten). **Validation set resmi** (193 label) dievaluasi terpisah di **Section 18** memakai ground truth yang disediakan panitia.
- Section EDA menghasilkan class distribution, descriptive statistics (dimensi, ukuran file, statistik piksel per channel) — langsung bisa dipakai untuk laporan Metodologi.
- Model: ResNet-18 pretrained ImageNet (transfer learning). Untuk arsitektur dari nol, ganti `build_model()` dengan implementasi manual residual block.
- Eksperimen hyperparameter di Section 15 sudah mencakup 3 jenis; boleh ditambah jenis lain dengan pola yang sama.